## 02. 전처리 로직 검증

전년동월대비 증감률 계산, 전세가율 조회, 거래량 집계 로직을 단계별로 검증합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part08_kb_dashboard.dashboard.db import execute_query
from projects.part08_kb_dashboard.dashboard.preprocessing import (
    get_yoy_data, get_jeonse_rate, get_trade_volume, get_date_range, get_all_indicators
)
from projects.part08_kb_dashboard.dashboard.constants import INDICATORS

### 1. 날짜 범위 확인

In [ ]:
start_y, start_m, end_y, end_m = get_date_range()
print(f"KB 매매가격지수 날짜 범위: {start_y}년 {start_m}월 ~ {end_y}년 {end_m}월")

### 2. 전년동월대비 증감률 계산 검증

`get_yoy_data()`는 현재 월 지수값과 12개월 전 지수값을 JOIN하여 증감률을 계산한다.

In [ ]:
# 2025년 9월 기준 매매 전년동월대비 증감률 (시군구별)
df = get_yoy_data('KB_매매가격지수', '가격지수', year=2025, month=9)
print(f"대상 시군구 수: {len(df)}개")
display(df.sort_values('증감률', ascending=False).head(10))

In [ ]:
# 증감률 분포 요약
print(df['증감률'].describe())

In [ ]:
# 전세 전년동월대비 증감률
df_jeonse_yoy = get_yoy_data('KB_전세가격지수', '가격지수', year=2025, month=9)
print(f"전세 YoY 대상 시군구 수: {len(df_jeonse_yoy)}개")
display(df_jeonse_yoy.sort_values('증감률', ascending=False).head(10))

### 3. 전세가율 조회 검증

`get_jeonse_rate()`는 KB_전세가율 테이블을 직접 조회한다. 단위는 % (0~100 범위).

In [ ]:
df_jeonse = get_jeonse_rate(year=2025, month=9)
print(f"전세가율 대상 시군구 수: {len(df_jeonse)}개")
print(f"값 범위: {df_jeonse['전세가율'].min():.1f}% ~ {df_jeonse['전세가율'].max():.1f}%")
display(df_jeonse.sort_values('전세가율', ascending=False).head(10))

### 4. 거래량 집계 검증

`get_trade_volume()`은 매매 테이블에서 직거래·취소 거래를 제외하고 시군구별로 집계한다.

In [ ]:
df_vol = get_trade_volume(year=2025, month=9)
print(f"거래량 집계 시군구 수: {len(df_vol)}개")
print(f"전국 총 거래량: {df_vol['거래량'].sum():,}건")
display(df_vol.sort_values('거래량', ascending=False).head(10))

### 5. 4개 지표 통합 조회

`get_all_indicators()`는 위 4개 함수를 한 번에 호출한다. 대시보드에서 사용하는 메인 함수다.

In [ ]:
all_data = get_all_indicators(year=2025, month=9)
for name, df in all_data.items():
    print(f"[{name}] {len(df)}개 시군구, 컬럼: {df.columns.tolist()}")

### 6. sig_cd 매핑 검증 — GeoJSON ↔ KB 데이터 연결

코로플레스 지도는 GeoJSON의 `sig_cd`(5자리)와 KB 데이터의 `sig_cd`(지역코드 앞 5자리)를 매칭한다.

In [ ]:
# KB sig_cd 형식 확인 (서울 지역)
df_seoul = all_data['매매 전년동월대비 증감률']
seoul_mask = df_seoul['지역명'].str.contains('서울|강남|마포|송파|강서', na=False)
print(df_seoul[seoul_mask][['sig_cd', '지역명', '증감률']].head(10).to_string(index=False))

In [ ]:
# 지표별 sig_cd가 일치하는지 확인 (INNER JOIN 가능 여부)
매매_sig = set(all_data['매매 전년동월대비 증감률']['sig_cd'])
전세_sig = set(all_data['전세 전년동월대비 증감률']['sig_cd'])
전세가율_sig = set(all_data['전세가율']['sig_cd'])
거래량_sig = set(all_data['매매 거래량']['sig_kor_nm'])  # 거래량은 지역명 기준

print(f"매매 YoY: {len(매매_sig)}개")
print(f"전세 YoY: {len(전세_sig)}개")
print(f"전세가율: {len(전세가율_sig)}개")
print(f"거래량(시군구명): {len(거래량_sig)}개")
print(f"매매∩전세 교집합: {len(매매_sig & 전세_sig)}개")